# Frequenzanalyse

[&larr; Modulübersicht](../00_INTRO.ipynb)

<figure style="text-align:center; margin:1em auto;">
    <img src="img/fft.svg" width=90%/>
</figure>

Die Frequenzanalyse ist ein sehr mächtiges Werkzeug in der Signalverarbeitung/Messtechnik, mit dem untersucht werden kann, welche **Frequenzen** in einem Signal vertreten sind. Die **Fouriertransformation** bildet dabei das Herzstück. Diese werden wir im Folgenden schrittweise kennenlernen, wobei zum einen das theoretische Hintergrundwissen und zum andern die praktische Anwendung erarbeitet werden. Für die Umsetzung in Python werden wir das Paket [SciPy](../05_SCIPY/01_SCIPY.ipynb) verwenden, welches wir bereits kennen.

---

## 🎯 Lernziele

- Du beherrscht die Sinus- bzw. Kosinusfunktion vollständig und kannst die Begriffe *Amplitude*, *Frequenz* (*Kreisfrequenz*), *Phasenverschiebung* sowie *Offset* (*Gleichanteil*) erklären.
- Du kannst die diskrete Fouriertransformation (DFT) für ein gegebenes Zeitsignal mithilfe des FFT-Algorithmus (SciPy) berechnen.
- Du kannst das Amplitudenspektrum grafisch darstellen und korrekt interpretieren.
- Du kannst das Abtasttheorem erläutern und kennst den Zusammenhang zwischen der Abtastfrequenz und der max. Frequenz im Spektrum.
- Du weisst, wie die Messdauer bzw. Sampleanzahl mit der Frequenzauflösung im Spektrum zusammenhängt.
- Du kannst den Begriff *Aliasing* korrekt erklären.
  
---

## 📋 Blockinhalt

Für die Einführung in die Frequenzanalyse/Fouriertransformation bzw. die selbstständige Einarbeitung stehen die folgenden Notebooks bereit:

1. Einführung/Theorie/Cheatsheet (vorliegendes Notebook)
2. Beispielaufgabe im Unterricht [02_FFT_Unterricht.ipynb](02_FFT_Unterricht.ipynb)
3. Übersicht FFT-Workflow [03_FFT_Workflow.ipynb](03_FFT_Workflow.ipynb)
4. Übungsaufgaben [11_FFT_Uebungen.ipynb](11_FFT_Uebungen.ipynb)

---

## 🛠 Voraussetzungen

Wir werden die theoretischen Grundlagen der Fourierreihe bzw. Fouriertransformation relativ zügig durchgehen, da die praktische Umsetzung im Vordergrund steht. Da Kosinus- bzw. Sinusschwingungen die Grundlage der Fouriertransformation bilden, wird die Darstellung einer Kosinus- bzw. Sinusschwingung im Zeitbereich, z. B. in der Form $y(t) = \hat y\cdot\cos(\omega t + \varphi)$ bzw. $y(t) = \hat y\cdot\sin(\omega t + \varphi)$ als bekannt vorausgesetzt. Die Begriffe *Amplitude*, *Frequenz* bzw. *Kreisfrequenz* sowie *Phasenverschiebung* sollten dir klar sein.

<figure style="text-align:center; margin:1em auto;">
    <img src="img/Sine.svg" />
</figure>

Es sollte mittlerweile klar sein, dass sich die Sinus- und die Kosinusfunktion nur durch eine Phasenverschiebung unterscheiden. Im Weiteren sollte bekannt sein, dass im Zusammenhang mit Schwingungen die Kreisfrequenz $\omega$ [rad/s] alternativ zur "normalen" Frequenz $f$ [Hz] verwendet wird und die beiden folgendermassen zusammenhängen:

$$\omega = 2 \pi \, f$$

☝🏼 Bsp.: Der Spannungsverlauf an einer 230V-Steckdose ist sinusförmig mit einer Amplitude von ca. 325 V und einer Frequenz von 50 Hz (oder 314 rad/s). Bei einem Dreiphasensystem ist die zweite Phase gegenüber der ersten um 120° (oder 2$\pi$/3 rad) verschoben und die dritte um 240° (oder 4$\pi$/3 rad). Woher kommt eigentlich die Angabe 230 V?

---

## 1. Motivation

In vielen technischen Problemstellungen treten Signale auf, welche unterschiedliche Frequenzanteile enthalten und bei der Analyse geht es u. a. um die Frage, welche Frequenzanteile wie stark ausgeprägt sind (Frequenzanalyse, auch Fourieranalyse oder harmonische Analyse genannt). In der Technik ist die Frequenzanalyse in unzähligen Gebieten anzutreffen, wobei für uns die Messtechnik im Vordergrund steht.

- Messtechnik (Schwingungen, akustische bzw. optische/bildgebende Verfahren)
- Datenkompression (Ton, Bild, Video etc.)
- Signalverarbeitung (Filterung, ...)
- Mathematik/Physik (Lösen von bestimmten Problemen)
- ...

## 2. Grundlagen

☝🏼 In diesem Abschnitt sind die theoretischen Grundlagen der Frequenzanalyse sehr kompakt anhand einiger Beziehungen zusammengestellt, die auf den ersten Blick kompliziert aussehen. Da für uns die praktische Anwendung der Frequenzanalyse im Vordergrund steht, musst du die theoretischen Grundlagen nicht bis ins letzte Detail verstehen.

### 2.1 Fourierreihe

**Gemäss *Fourier* kann jede beliebige periodische Funktion $y(t)$, nachfolgend Signal oder Zeitsignal genannt, als Summe von unendlich vielen Sinus- und Kosinusschwingungen dargestellt werden (Fourierreihe)**. Wie wir später sehen werden, besteht das Ziel der Fouriertransformation im Wesentlichen darin, die Koeffizienten (Amplituden und Phasen) der einzelnen Schwingungen zu bestimmen, d. h. zu ermitteln wie ausgeprägt bestimmte Frequenzanteile auftreten.

#### **Fourierreihe in Sinus-Kosinus-Form**

$$ y(t) = y_0 +\sum_{k=1}^{\infty}\left[a_{k}\cos\left(\omega_k \, t\right)+b_{k}\sin\left(\omega_k \, t\right)\right] $$

☝🏼 Ein Messsignal hat im Allgemeinen einen Offset (auch Gleichanteil oder DC-Anteil) genannt - dies entspricht dem Mittelwert des Signals. Der Gleichanteil wird hier mit $y_0$ bezeichnet.

☝🏼 Die niedrigste Frequenz in einem periodischen Signal wird Grundfrequenz genannt und Vielfache davon Oberschwingungen (1. Oberschwingung hat doppelte Frequenz, 2. Oberschwingung hat dreifache Frequenz etc.).

$$ \omega_k = 2 \pi \frac{k}{T} \quad \textsf{mit Periodendauer } T$$

#### **Fourierreihe in Amplituden-Phasen-Form**

Da die Sinusfunktion lediglich eine verschobene Kosinusfunktion ist, kann die obige Beziehung auch kompakter mit Amplitude $A$ und Phasenverschiebung $\varphi$ geschrieben werden. 

$$ y(t) = y_0 +\sum_{k=1}^{\infty}A_{k}\cos\left(\omega_k \, t-\varphi_{k}\right) $$

Dabei zeigt sich schön, worin letztlich das Ziel der Fouriertransformation besteht.

- Bestimmen der Amplituden $A$ für die verschiedene Frequenzen im Signal $\,\to\,$ **Amplitudenspektrum**
- Bestimmen der Phasenverschiebungen $\varphi$ für die verschiedene Frequenzen im Signal $\,\to\,$ **Phasenspektrum**

Amplituden- und Phasenspektrum werden zusammengenommen **Frequenzspektrum** genannt, wobei für uns i. Allg. nur das Amplitudenspektrum relevant ist. Die folgende Animation zeigt das Prinzip der Fourierreihe und des Frequenzspektrums sehr anschaulich.

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FFT.gif" width=60% />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Fouriertransformation und Frequenzbereich <a href="https://en.wikipedia.org/wiki/File:Fourier_transform_time_and_frequency_domains.gif">[Quelle]</a>
    </figcaption>
</figure>

#### **Fourierreihe in Exponentialform**

Am kompaktesten wird die Fourierreihe in der Exponentialform dargestellt. Mithilfe der *Eulerschen Formel* $e^{j \omega t} = \cos(\omega t) + j \cdot \sin(\omega t)$ kann die Fourier-Reihe mit der komplexen Exponentialfunktion geschrieben werden. Die Koeffizienten $Y_k$ stellen dabei die *komplexen Amplituden* dar und sind die gesuchten Fourierkoeffizienten.

$$ y(t)=\sum_{k\in\mathbb{Z}}Y_{k} \cdot e^{j \omega_k \,t} $$

☝🏼 Es ist üblich, die Fouriertransformierte eines Signals mit dem entsprechenden Grossbuchstaben des Zeitsignals zu bezeichnen. Wir bezeichnen das Zeitsignal in der Regel mit `y` und dementsprechend die Transformierte mit `Y`. Der Betrag (Amplitude) der komplexen Grösse `Y` wird mathematisch mit $|Y|$ geschrieben, in Python verwenden wir als Variablenname dafür `A = abs(Y)` (A für Amplitude).

### 2.2 Diskrete Fouriertransformation (DFT)

Wie oben angedeutet wird ein Signal aus dem Zeitbereich mithilfe der Fouriertransformation in den Frequenzbereich (auch Bildbereich genannt) transformiert. Konkret bedeutet dies, die entsprechenden Fourierkoeffizienten zu berechnen. Wenn dies numerisch für ein zeitdiskretes (abgetastetes) Signal $y[n]$ gemacht wird, spricht man von der **diskreten Fouriertransformation** (DFT). Die Fourierkoeffizienten $Y_k$ sind komplex und beinhalten die Amplitudeninformationen (Betrag) und Phaseninformationen (Winkel).

$$ Y[k] = \sum_{n=0}^{N-1} y[n] \cdot e^{-j \omega_k \, n} $$

### 2.3 FFT-Algorithmus

Die Berechnung gemäss obiger Beziehung erfordert für lange und hochaufgelöste Zeitsignale einen erheblichen Rechenaufwand. Erst die Entwicklung des **FFT-Algorithmus** (*Fast Fourier Transform*) ermöglichte die effiziente Berechnung der diskreten Fourier-Transformation (DFT) und somit den breiten Einsatz in technischen Anwendungen.

#### 2.3.1 SciPy-Implementierung (`scipy.fft`)

SciPy stellt das Modul **fft** zur Verfügung, welches Funktionen zur Berechnung der Fouriertransformation enthält. Im Wesentlichen ist das die Funktion `fft()`, doch es gibt eine Reihe verwandter Funktionen, von denen wir einige verwenden werden. Die Funktion `fft()` ist für beliebige (auch komplexe) Signale definiert und liefert auch Frequenzanteile für negative Frequenzen zurück. Für reelle Zeitsignale (mit denen wir es zu tun haben), ist das Frequenzspektrum immer symmetrisch, d.h. die Frequenzanteile der negativen Frequenzen sind immer identisch mit denjenigen der positiven Frequenzen und beinhalten daher keine zusätzlichen Informationen. Wir können diese negative Frequenzachse also problemlos ignorieren (hat physikalisch keine Bedeutung). Deshalb werden wir anstelle der allgemeinen Funktion `fft()` stets die speziell für reelle Signale geeignete Funktion `rfft()` verwenden (und für die Erzeugung der Frequenzachse die Funktion `rfftfreq()`).

## 3. Praktische Umsetzung

Die Berechnung des Frequenzspektrums (= Amplituden- und Phasenspektrums) mithilfe des FFT-Algorithmus ist dank der oben erwähnten SciPy-Implementierung sehr einfach, doch es gibt einige wichtige Aspekte, die beachtet werden müssen:

&rarr; <a href=#frequency>[3.1 Frequenzauflösung und max. Frequenz](#3.1-Frequenzauflösung-und-maximale-Frequenz)</a><br>
&rarr; <a href=#windowing>[3.2 Periodizität bzw. Fensterfunktionen](#3.2-Periodizität-bzw.-Fensterfunktionen)</a><br>
&rarr; <a href=#samples>[3.3 Sampleanzahl](#3.3-Sampleanzahl)</a><br>
&rarr; <a href=#aliasing>[3.4 Aliasing bzw. Abtasttheorem](#3.4-Abtasttheorem)</a><br>

### 3.1 Frequenzauflösung und maximale Frequenz 

<a name=frequency></a>Die nachfolgende Abbildung zeigt ein mit 1600 Hz abgetastetes Signal, wobei es sich um eine 50 Hz-Schwingung handelt. Darunter ist das entsprechende Amplitudenspektrum dargestellt, welches erwartungsgemäss einen Peak bei 50 Hz aufweist (alle anderen Frequenzen haben keinen Anteil, da es sich beim Signal um eine reine 50 Hz-Schwingung handelt). Die Höhe des Peaks (Amplitude) entspricht der Schwingungsamplitude im Zeitsignal ($= 1$).

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_1.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Zeitsignal (oben) und Amplitudenspektrum (unten) einer 50 Hz-Schwingung
    </figcaption>
</figure>

☝🏼 Wenn du das obige Diagramm lesen und korrekt interpretieren (also komplett verstehen) kannst, hast du das Grundlegendste der Frequenzanalyse bereits verstanden. Gehe erst weiter, wenn dies der Fall ist.

Anhand der obigen Abbildung können an dieser Stelle folgende Zusammenhänge erkannt werden:

- Die Frequenzauflösung $\Delta f$ im Spektrum entspricht 25 Hz, was gerade dem Kehrwert der Dauer der Messung $t_{meas} =$ 0.04 s entspricht. Dies ist intuitiv verständlich, denn die "langsamste" Schwingung, die in einem Zeitfenster von 0.04 s detektiert werden kann, entspricht einer 25 Hz-Schwingung. Für eine hohe Auflösung im Frequenzbereich ist also eine lange Messdauer nötig. Die Frequenzauflösung ist gleichbedeutend mit der kleinsten Frequenz, welche im Spektrum angezeigt wird (abgesehen vom Gleichanteil bei 0 Hz). Da sich die Messdauer mit $t_{meas} = N \cdot 1/f_S$ ausdrücken lässt, kann die Frequenzauflösung auch mit $f_S / N$ berechnet werden.
  
  $$ \Delta f = \frac{1}{t_{meas}} = \frac{f_S}{N}$$

- Die maximale Frequenz im Spektrum entspricht $f_{max} =$ 800 Hz, was gerade der Hälfte der Abtastfrequenz $f_S =$ 1600 Hz entspricht. Dies ist ebenfalls recht anschaulich erklärbar. Um eine Sinusschwingung überhaupt noch als Schwingung detektieren zu können, braucht es mindestens 2 Abtastungen pro Periode, d. h. im vorliegenden Fall kann mit einer Abtastrate von 1600 Hz höchstens eine 800 Hz-Schwingung erkannt werden. Die maximale Frequenz im Spektrum $f_{max}$ wird auch *Nyquist*-Frequenz genannt (vgl. <a href=#aliasing>Abtasttheorem</a>).
  
  $$ f_{max} = \frac{f_S}{2} $$

Die relevanten Grössen im Frequenzbereich ($f_S$, $\Delta f$, $f_{max}$) hängen direkt mit den Grössen im Zeitbereich ($\Delta t$, $t_{meas}$, $N$) zusammen.

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_1a.svg" width=100% />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Zusammenhang zwischen Zeit- und Frequenzbereich
    </figcaption>
</figure>

### 3.2 Periodizität & Fensterfunktionen 

<a name=windowing></a>Im obigen Beispiel entspricht die Dauer der "Messung" genau einem Vielfachen der Periodendauer der Schwingung, d. h. das Signal ist exakt periodisch im betrachteten Ausschnitt ("Fenster") - gedanklich kann der Anfang des Signals wieder ans Ende "angeschlossen" werden, ohne dass ein Sprung bzw. eine Unstetigkeit entsteht. Nachfolgend ist das gleiche Signal wie oben dargestellt, dabei wurde nun aber etwas länger gemessen ($t_{meas} =$ 0.05 s), d. h. die Messzeit entspricht nicht mehr genau einem Vielfachen der Periodendauer der Schwingung.

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_2.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Messzeit entspricht nicht genau einem Vielfachen der Schwingungsperiode
    </figcaption>
</figure>

Auffällig ist dabei folgendes:
- Obwohl es sich um die genau gleiche reine 50 Hz-Schwingung wie in der ersten Abbildung handelt, treten nun im Spektrum auch Anteile anderer Frequenzen auf.

Die Erklärung dafür liegt darin begründet, dass die Fouriertransformation von __periodischen Signalen__ ausgeht und diese Voraussetzung hier offensichtlich verletzt wurde. Denkt man sich das obige Signal periodisch weiter, so resultiert das nachfolgend abgebildete Zeitsignal und es ist ersichtlich, dass nun "Sprünge" auftreten, welche das Amplitudenspektrum "verfälschen". Die Sprünge führen zu zusätzlichen (unphysikalischen) Frequenzanteilen und "verschmieren" das Spektrum. Dieser Effekt wird auch Leck-Effekt bzw. *Spectral Leakage* genannt.

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_3.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: "Gedankliche" periodische Verlängerung des Zeitsignals führt zu Sprüngen an den Übergangsstellen
    </figcaption>
</figure>

Wie könnte dieses Problem nun behoben werden? Eine Möglichkeit, die sich in der Praxis etabliert hat, besteht darin, sogenannte [Fensterfunktionen](https://de.wikipedia.org/wiki/Fensterfunktion) zu verwenden, mit denen das Zeitsignal vor der Fouriertransformation dahingehend verändert wird, dass solche Sprünge nicht oder nicht mehr so ausgeprägt auftreten. Man kann sich das so vorstellen, dass das Signal am Anfang und am Ende "langsam auf 0 heruntergezogen wird", sodass ein glatter Übergang zwischen Ende und Anfang erreicht wird bei der periodischen Wiederholung.

In nachfolgender Abbildung sind typische Fensterfunktionen dargestellt, wobei alle gemeinsam haben, dass sie am Anfang und am Ende gegen 0 oder einen kleinen Wert streben, womit das oben beschriebene Problem entschärft wird. Diese Fensterfunktionen sind unter `signal.windows` in SciPy verfügbar.

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_4.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Typische Fensterfunktionen
    </figcaption>
</figure>

Die Anwendung der Fensterfunktion ist denkbar einfach. Das Messignal wird elementweise mit der (gleichlangen) Fensterfunktion multipliziert und geht so am Anfang und am Ende gegen 0.

☝🏼 Der Gleichanteil (Offset) des Signals muss für die Multiplikation mit der Fensterfunktion entfernt und danach wieder hinzugefügt werden.

In nachfolgender Abbildung ist nochmals das "gedanklich" periodisch wiederholte Signal dargestellt, nun wurde es aber mit der *Tukey*-Fensterfunktion konditioniert. Man erkennt, dass nun keine Sprünge zwischen den periodischen Wiederholungen auftreten, da das Signal an diesen Stellen jeweils 0 ist. 

☝🏼 Da wir in diesem Beispiel lediglich ca. 2 Perioden der Schwingung erfassen, wird das Signal durch die Fensterfunktion stark verändert. In der Praxis bzw. einem "realen" Signal ist die Messzeit im Vergleich zur Periodendauer der Schwingung(en) in der Regel sehr gross und dieser Effekt daher vernachlässigbar.

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_5.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Periodische Verlängerung des Zeitsignals mit Fensterung, d.h. es treten keine Sprünge mehr au
    </figcaption>
</figure>

Betrachten wir nun das Amplitudenspektrum des mit der Fensterfunktion vorkonditionierten Signals verschwinden die unphysikalischen hohen Frequenzen weitgehend aus dem Spektrum (vgl. Anmerkung oben).

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_6.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Amplitudenspektrum mit/ohne Fensterfunktion
    </figcaption>
</figure>

### 3.3 Sampleanzahl 

<a name=samples></a>Die Berechnung des Frequenzspektrums ist für lange Messungen, welche mit einer hohen Abtastfrequenz abgetastet wurden (also aus sehr vielen Samples bestehen), sehr aufwändig und der FFT-Algorithmus nutzt spezielle Eigenschaften (Symmetrien etc.) des Frequenzspektrums aus um die Berechnung möglichst effizient zu machen. Am effizientesten ist der Algorithmus, wenn die Sampleanzahl $N$ einer 2er-Potenz entspricht (also z. B. $2^{16}$ oder $2^{20}$ usw.). Da Messungen in der Praxis in der Regel nicht genau eine solche Anzahl Samples aufweisen, gibt es verschiedene Möglichkeiten:

- Die meisten modernen FFT-Implementierungen (z. B. SciPy, NumPy, etc.) können relativ problemlos mit einer "unpassenden" Sampleanzahl umgehen. Allenfalls dauert die Berechnung etwas länger, doch das ist bei Berechnungen auf einem modernen Computer etc. meistens nicht problematisch. Bei Realtime-Berechnungen auf leistungsschwachen Geräten (Mikrocontrollern etc.) wird die Sampleanzahl allerdings idealerweise günstig gewählt.
- Die FFT-Funktion in SciPy hat einen zweiten Parameter (`n`), mit dem man steuern kann, wie viele Samples für die FFT-Berechnung verwendet werden. Ist `n` kleiner als die Sampleanzahl des Signals, so werden nur `n` Samples verwendet und die übrigen ignoriert (*Cropping*). Ist `n` grösser als die Sampleanzahl des Signals, so wird das Signal mit 0 aufgefüllt (*Zero Padding*). Dies hat einen geringfügigen Einfluss auf das Spektrum, doch dieser ist in vielen praktischen Anwendungen unerheblich. Nachfolgend ist das Spektrum einer verrauschten Sinusschwingung dargestellt, wobei einmal die ursprüngliche (ungünstige) Sampleanzahl $N$ und dann die nächstkleinere bzw. nächstgrössere 2er-Potenz als Sampleanzahl verwendet wurde. Darunter sind die Rechenzeiten der FFT angegeben und man erkennt, dass die ungünstige Sampleanzahl eine 5..10 x längere Rechenzeit aufweist. Für uns ist dies jedoch im Allgemeinen unerheblich, d. h. wir werden üblicherweise die effektive Sampleanzahl der Messung verwenden.

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_7.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Einfluss der verwendeten Sampleanzahl auf das Amplitudenspektrum
    </figcaption>
</figure>

| Sampleanzahl $N$              |  Zeitbedarf FFT [ms]  |
| -----------                   | -----------           |
| 3839573                 | 1131                  |
| $2^{21}$ = 2097152 (Cropping) | 98                    |
| $2^{22}$ = 4194304 (Padding)  | 293                   |

- Eine weitere Möglichkeit besteht darin, das Signal zu "resamplen", d. h. das Signal an anderen Stützstellen abzutasten, um so eine günstige Sampleanzahl zu erreichen. Dies kann z. B. sehr einfach mittels Interpolation (vgl. `interp()` aus NumPy) erfolgen.

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_8.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Interpoliertes Zeitsignal, um eine günstige Sampleanzahl zu erreichen
    </figcaption>
</figure>

### 3.4 Abtasttheorem

<a id=aliasing></a>Wir haben oben gesehen, dass die maximale Frequenz $f_{max}$ im Spektrum der halben Abtastfrequenz $f_S$ entspricht. Dies bedeutet, dass Schwingungen bis zu dieser Frequenz vollständig aus dem abgetasteten Signal rekonstruiert werden können. Dies ist ein grundlegendes Theorem der Informationstheorie und wird *Nyquist-Shannon-Abtasttheorem* (oder kurz **Abtasttheorem**) genannt. 

> Ein Signal muss mit einer Abtastfrequenz $f_S$ abgetastet werden, die mehr als doppelt so gross ist wie die höchste im Signal auftretende Frequenz $f_{\mathrm{max}_{Signal}}$.

$$ f_S > 2 \cdot f_{\mathrm{max}_{Signal}} $$

Die Verletzung des Abtasttheorems kann zu einer kompletten Verfälschung des Spektrums führen, da "scheinbare" Frequenzanteile erzeugt werden, die aber nicht im Signal enthalten sind. An einem Beispiel soll gezeigt werden, was passiert, wenn ein Signal mit einer zu geringen Abtastfrequenz abgetastet wird. In nachfolgender Abbildung wird eine 25 Hz-Schwingung (blau) mit einer zu kleinen Abtastfrequenz von $f_S =$ 30 Hz erfasst, womit das Abtasttheorem verletzt wird (30 Hz $\ngtr$ 2 $\cdot$ 25 Hz). Wird vom erfassten Signal nun das Frequenzspektrum erstellt, erscheint dort ein Peak bei 5 Hz, obwohl im Signal weit und breit keine 5 Hz-Schwingung vorhanden ist. Zum Vergleich ist zusätzlich der korrekte Peak bei 25 Hz eingetragen (blau), der mit korrekter Abtastung erkannt worden wäre. Dieses Phänomen wird als *Aliasing* bezeichnet und betrifft sämtliche Abtastvorgänge.

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_9.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Abtastung mit zu kleiner Abtastfrequenz (rote Kurve) führt zu unphysikalischen Frequenzanteilen (Aliasing)
    </figcaption>
</figure>

☝🏼 Man muss sich im Klaren sein, dass im obigen Diagramm zum Vergleich das "korrekte" Signal in blau eingetragen ist. Dieses würde man bei einer realen Messung mit der zu tiefen Abtastfrequenz aber eben genau nicht erkennen, sondern nur das rote ("falsche") Signal und es gäbe im Nachhinein keine Möglichkeit, dieses Problem zu erkennen bzw. zu korrigieren. Die Einhaltung des Abtasttheorems ist daher extrem wichtig!

Zum Beispiel tritt dieses Phänomen häufig in Videos auf, wenn eine Bewegung gefilmt wird, die "schneller" ist, als es die Abtastung (zeitliche Auflösung) erfassen kann. Die Bewegung scheint dann z. T. viel zu langsam oder sogar in die "verkehrte" Richtung abzulaufen ("Rad dreht rückwärts" etc.). Es scheint dann eine "Frequenz" zu geben, die in Wirklichkeit natürlich nicht vorkommt.

<figure style="text-align:center; margin:1em auto;">
    <video controls src="img/Aliasing.mp4" type="video/mp4" width='100%' >
    </video>
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Aliasing-Effekt in Video <a href="https://www.youtube.com/watch?v=ByTsISFXUoY&ab_channel=FABIOGERMANGUERREROMORENO">[Quelle]</a>
    </figcaption>
</figure>

---

## 💡 Verständnisfragen

- Was ist Sinn und Zweck der Frequenzanalyse? Könntest du das jemandem erklären, der nichts mit Technik/Mathematik etc. am Hut hat?
- Was wird im Amplitudenspektrum dargestellt?
- Erkläre das Abtasttheorem in eigenen Worten?
- Warum wird eine Fensterung (*Windowing*) durchgeführt?

---

## A. Anhang

> ☝🏼 Hier folgen weiterführende Informationen zur Frequenzanalyse, die allerdings nicht zum Stoffumfang des Moduls gehören und daher als optional zu betrachten sind.

Für alle unsere behandelten Signale gilt, dass die Frequenzen konstant sind, d. h. die Frequenzen ändern sich nicht über die Zeitdauer. In der Praxis ist dies in vielen Fällen so bzw. die Messungen werden so durchgeführt, dass dies erreicht wird (Messung in "stationärem" bzw. "eingeschwungenem" Zustand).

Es gibt aber Fälle - stelle dir z. B. Vibrationen beim Hochlauf (Anfahren) eines Motors vor - in denen sich die Frequenzen während der Zeit (Messdauer) ändern. Als Beispiel ist nachfolgend ein Signal gezeigt, bei dem die Frequenz mit der Zeit laufend zunimmt. Am Anfang der Messung schwingt das Signal mit 5 Hz und anschliessend nimmt die Frequenz laufend zu bis am Ende der Messung 25 Hz vorliegen.

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_10.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Signal mit zeitlich veränderlicher Frequenz (Frequenz nimmt laufend zu)
    </figcaption>
</figure>

Da die Fouriertransformation (FFT) zeitliche Veränderungen nicht berücksichtigt, ist im Amplitudenspektrum (siehe nachfolgende Abbildung) nicht ersichtlich, dass die Frequenz am Anfang der Messung tiefer ist und mit zunehmender Messdauer zunimmt. Man erkennt zwar, dass Frequenzen zwischen 5 und 25 Hz vorhanden sind, doch nicht viel mehr (die FFT mittelt ("verschmiert") die zeitlich veränderliche Frequenz sozusagen).

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_11.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Amplitudenspektrum des obigen Zeitsignals
    </figcaption>
</figure>

Bei solchen Signalen ist es sinnvoll, die FFT nicht auf das gesamte Signal anzuwenden, sondern auf kleinere Zeitintervalle (z. B. von 0-1 s, dann von 1-2 s etc.). Man spricht dann von der **Kurzzeit-Fouriertransformation** (*Short Time Fourier Tranform*) STFT. Dabei resultiert nun jeweils ein Amplitudenspektrum für die unterschiedlichen Zeitintervalle, d. h. man kann nun sozusagen einen zeitlichen Verlauf des Amplitudenspektrums anschauen und die Amplituden nicht nur über die Frequenz sondern gleichzeitig auch über die Zeit betrachten (siehe nachfolgende Abbildung). Man sieht nun schön, wie sich der Frequenzpeak über die Zeit zu höheren Frequenzen verschiebt (ca. 5 Hz am Anfang der Messung und ca. 25 Hz am Ende der Messung).

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_12.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Kurzzeit-Fouriertransformation (Amplitude gegenüber Frequenz und Zeit)
    </figcaption>
</figure>

Die obige Abbildung zeigt schön, dass nun neben der (gewohnten) Frequenzachse auch eine Zeitachse vorhanden ist und man daher zeitliche Veränderung des Spektrums erkennen kann. Diese Darstellung ist intuitiv gut verständlich, doch 3D-Plots sind generell schwierig zu lesen (besonders wenn man sie nicht interaktiv drehen kann). Daher ist die nachfolgende Abbildung in der Regel besser geeignet. Sie enthält dieselben Informationen wie der obige 3D-Plot, doch visualisiert die Amplitude mit der Farbachse. Dieses Diagramm wird Spektrogramm (*Spectrogram*) genannt.

<figure style="text-align:center; margin:1em auto;">
    <img src="img/FA_13.svg" />
    <figcaption style="font-size:0.9em; font-style:italic; margin-top:0.4em;">
        Abb.: Kurzzeit-Fouriertransformation (Spektrogramm)
    </figcaption>
</figure>

Die Frequenz ist dabei auf der vertikalen Achse und die Zeit auf der horizontalen Achse aufgetragen und man kann sehr gut ablesen, dass die Frequenz bei ca. 5 Hz startet und dann linear bis auf ca. 25 Hz hochläuft bis zum Ende der Messung.